# ⚡ Module 3: GPU-Accelerated Astronomy with CuPy

Welcome to the heavy-hitters module! You have two NVIDIA cards in your fleet: a **3050Ti** (WSL) and a **1050Ti** (Ubuntu).

Astronomical images can be very large (up to 50,000 x 50,000 pixels). Processing these on a CPU is slow. GPUs, however, have thousands of small cores designed for exactly this kind of math.

---

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt

try:
    import cupy as cp
    GPU_AVAILABLE = True
    print("🚀 NVIDIA GPU detected! Ready for acceleration.")
except ImportError:
    GPU_AVAILABLE = False
    print("💻 GPU not found. Please run this on your Windows WSL or Ubuntu GPU node.")

## 1. The Performance Test (CPU vs. GPU)
Let's create a huge 4000x4000 grid of 'star' data and see how long it takes to calculate the average brightness across the whole field.

In [ ]:
size = 4000
print(f"🔢 Generating a {size}x{size} mock star field...")
data_cpu = np.random.random((size, size)).astype(np.float32)

# CPU Benchmark
start = time.time()
mean_cpu = np.mean(data_cpu)
end = time.time()
print(f"🐢 CPU Time: {end - start:.4f} seconds")

if GPU_AVAILABLE:
    # GPU Benchmark
    # First, transfer data to GPU VRAM
    data_gpu = cp.array(data_cpu)
    
    start = time.time()
    mean_gpu = cp.mean(data_gpu)
    # Force GPU to finish before stopping clock
    cp.cuda.Stream.null.synchronize()
    end = time.time()
    print(f"⚡ GPU Time: {end - start:.4f} seconds")
    
    speedup = (end_cpu - start_cpu) / (end_gpu - start_gpu) # Approximate
    print(f"🏎️ Speedup: ~{speedup:.1f}x")

## 2. Removing 'Noise' (Cosmic Ray Filtering)
Deep-space images often have bright 'speckles' from high-energy particles hitting the sensor. A common way to remove them is using a **Median Filter**.

The 3050Ti in your WSL machine is perfect for this!

In [ ]:
if GPU_AVAILABLE:
    from cupyx.scipy.ndimage import median_filter
    
    print("🧹 Filtering out cosmic rays using GPU...")
    start = time.time()
    filtered_gpu = median_filter(data_gpu, size=5)
    cp.cuda.Stream.null.synchronize()
    print(f"✅ Filtered {size}x{size} image in {time.time() - start:.4f} seconds.")

## 3. Important: Memory Management
Your **3050Ti** and **1050Ti** have limited Video RAM (VRAM). Unlike the CPU, which can use your system's 16GB or 32GB of RAM, the GPU can only process what fits in its own memory.

**Tips for your Lab:**
1. **Chunking**: If you have a massive image, break it into 1000x1000 chunks.
2. **Clearing Cache**: Use `cp.get_default_memory_pool().free_all_blocks()` to release memory between tasks.